# FASTopic on new data
FASTopic on 13 datasets: the 5 from the paper, DBpedia (in place of WoS), 5 existing labelled datasets the paper did not use, and 2 we built ourselves (Guardian, TMDB). Section 1 compares the datasets, Section 2 shows how we built ours, Section 3 shows what the data looks like after preprocessing, and Sections 4–5 give FASTopic's results.

In [13]:
import json
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("..")
R, D = Path("results"), Path("datasets")
M6 = ["CV", "TD", "Purity", "NMI", "Acc", "F1"]
GROUPS = {"paper": ["20NG", "NYT", "NeurIPS", "ACL", "Wikitext-103"],
          "replaces WoS": ["DBpedia"],
          "existing, new to FASTopic": ["BBC", "OHSUMED", "Reuters", "TREC-coarse", "TREC-fine"],
          "built by us": ["Guardian", "TMDB"]}

cv = pd.read_csv(R / "cv_palmetto.csv").drop_duplicates(["dataset", "K", "seed"])

def load(name):
    df = pd.read_csv(R / name).drop_duplicates(["dataset", "K", "seed"], keep="first")
    return df.merge(cv, on=["dataset", "K", "seed"], how="left")

def texts(name):
    return [l for s in ("train", "test") for l in (D / name / f"{s}_texts.txt").read_text().splitlines()]

## 1. Datasets
Statistics after preprocessing (train + test).

In [14]:
def stats(name):
    docs = texts(name)
    labels = D / name / "train_labels.txt"
    return {"docs": len(docs),
            "avg words": round(np.mean([len(d.split()) for d in docs]), 1),
            "vocab": len((D / name / "vocab.txt").read_text().split()),
            "labels": len(np.unique(np.loadtxt(labels, dtype=int))) if labels.exists() else "–"}

pd.DataFrame.from_dict({(g, n): stats(n) for g, names in GROUPS.items() for n in names}, orient="index")

docs  avg words  vocab labels
paper                     20NG          18846      109.0   5000     20
                          NYT            9172      175.3  10000     12
                          NeurIPS        7237     2080.8  10000      –
                          ACL           10560     2018.1  10000      –
                          Wikitext-103  28532     1355.8  10000      –
replaces WoS              DBpedia       24999       20.4  10000     14
existing, new to FASTopic BBC            2225      176.3  10000      5
                          OHSUMED       18302       96.9  10000     23
                          Reuters        9160       61.3  10000     58
                          TREC-coarse    5948        4.0   8137      6
                          TREC-fine      5948        4.0   8137     50
built by us               Guardian       7306      324.8  10000      6
                          TMDB           3206      105.4  10000      6

**Notes:** The datasets differ in length (4 words for TREC questions, about 2,000 for NeurIPS and ACL papers) and in number of labels (5 for BBC, 58 for Reuters). TREC-coarse and TREC-fine are the same questions with two label sets, 6 broad and 50 fine.

## 2. Building Guardian and TMDB
### 2.1. What we collected
One raw record from each source, as saved by the collection scripts (long text cut to 300 characters). TMDB usernames are not shown.

In [15]:
def first_record(path):
    with open(path) as f:
        rec = json.loads(f.readline())
    return {k: (v[:300] + " ...") if isinstance(v, str) and len(v) > 300 else v
            for k, v in rec.items() if "author" not in k}                     # never show reviewer names

for path in [D / "raw/guardian/sport.jsonl", D / "raw/tmdb/films.jsonl", D / "raw/tmdb/reviews.jsonl"]:
    print(path.relative_to(D))
    display(pd.Series(first_record(path)).to_frame("value"))

raw/guardian/sport.jsonl


,value
id,sport/2025/jan/01/sam-konstas-australia-cricke...
section,sport
date,2025-01-01T06:26:11Z
title,Sam Konstas’ stunning Australia debut not his ...
wordcount,519
text,The impact of young opener Sam Konstas on the ...


raw/tmdb/films.jsonl


,value
film_id,1423191
title,Resident Evil
release_date,2026-09-16
genres,"[Horror, Science Fiction, Adventure]"
sampled_under,Horror
n_reviews,9


raw/tmdb/reviews.jsonl


,value
review_id,6aaea8027d78cf7d7666f72e
film_id,1423191
rating,None
created_at,2026-09-19T15:19:30Z
text,"This film is good, i really like the cast and ..."


### 2.2. From raw records to dataset
Documents left after each step.

In [16]:
for name in ["guardian", "tmdb"]:
    print(name)
    display(pd.read_csv(R / "construction" / f"{name}_steps.csv"))

guardian


,step,docs
0,raw articles collected,13616
1,in the six chosen labels,13104
2,after removing duplicates,13099
3,after removing roundups / quizzes,12272
4,after minimum length (150 words),12252
5,after capping at 1500 per label,7306


tmdb


,step,docs
0,raw reviews collected,4837
1,films with exactly one of the six genres,3685
2,after removing duplicates,3681
3,after minimum length (30 words),3210
4,after keeping English only,3206


### 2.3. Labels

In [17]:
for name in ["guardian", "tmdb"]:
    print(name)
    display(pd.read_csv(R / "construction" / f"{name}_labels.csv"))

guardian


,label,docs
0,sport,1500
1,australia-news,1500
2,world,1500
3,culture,1414
4,lifeandstyle,808
5,environment,584


tmdb


,label,docs
0,Science Fiction,1152
1,Comedy,570
2,Horror,558
3,War,375
4,Western,304
5,Romance,247


### 2.4. Before and after preprocessing
The same document as collected and as FASTopic sees it (lowercased, stopwords, numbers and rare words removed). The processed version is found by word overlap with the raw text.

In [18]:
def raw_text(rec):
    return max((v for k, v in rec.items() if isinstance(v, str) and "author" not in k), key=len)

def before_after(raw_files, name, n_words=60):
    processed = [(d, set(d.split())) for d in texts(name)]
    for path in raw_files:
        for line in open(path):
            raw = raw_text(json.loads(line))
            words = set(re.findall(r"[a-z]+", raw.lower()))
            doc, overlap = max(((d, len(words & s) / len(words | s)) for d, s in processed), key=lambda x: x[1])
            if overlap > 0.3:                                                 # this record made it into the dataset
                print("BEFORE:", " ".join(raw.split()[:n_words]), "...\n")
                print("AFTER: ", " ".join(doc.split()[:n_words]), "...")
                return

labels = pd.read_csv(R / "construction" / "guardian_labels.csv").label
before_after([D / f"raw/guardian/{l}.jsonl" for l in labels], "Guardian")

BEFORE: The impact of young opener Sam Konstas on the Boxing Day Test surprised fans as much as the India bowlers, and Australian wicketkeeper Alex Carey says even he underestimated how much of a difference the 19-year-old would make. Australia trained today in Sydney ahead of Friday’s fifth and final Test against India at the SCG. The home side needs only ...

AFTER:  impact young opener sam konstas boxing day test surprised fans india bowlers australian alex carey says underestimated difference year old australia trained today sydney ahead fifth final test india scg home needs draw claim border trophy dramatic victory fourth test melbourne aided konstas debutant lay platform test quick innings saw striker bowler new south action fielding short leg home wrapped ...


In [19]:
before_after([D / "raw/tmdb/reviews.jsonl"], "TMDB")

BEFORE: Curry Barker has crafted something mostly special for an almost non-existent budget. The story is compelling, the acting is solid, and Obsession feels like a breath of fresh air in comparison to similar horror films. However, it’s disappointing that one shrill character trait nearly ruins the entire film. It’s because of this that Obsession will have to settle for being ...

AFTER:  curry barker crafted special non existent budget story compelling acting solid obsession feels like breath fresh air comparison similar horror films disappointing character trait nearly ruins entire film obsession settle written film dominated irritating character future like frank ...


## 3. What every dataset looks like after preprocessing
First test document of each dataset (first 25 words).

In [20]:
for names in GROUPS.values():
    for n in names:
        doc = (D / n / "test_texts.txt").read_text().splitlines()[0]
        print(f"{n:13} {' '.join(doc.split()[:25])} ...")

20NG          neil subject need info organization university buffalo lines news software vax vms vnews nntp posting host little confused models heard someone tell differences far features ...
NYT           rev freed humanitarian grounds sentenced hard labor plotting north korea north korea canadian pastor sentenced life hard labor seoul south korea canadian pastor sentenced hard ...
NeurIPS       independent component analysis identification artifacts recordings oja lab computer info science helsinki university technology box fin finland brain research unit low temperature lab helsinki university ...
ACL           adaptive chinese word segmentation gao chang ning huang xia microsoft research beijing institute technology beijing university beijing university work done xia visiting microsoft research asia ...
Wikitext-103  robert robert english film television theatre actor guest starring role television series bill followed starring role play written simon performed royal court theatre 

## 4. FASTopic results ($K = 50$)
Mean ± SD over seeds. Wikitext-103, NeurIPS and ACL have no labels, so no clustering or classification scores.

In [21]:
res = pd.concat([load(f) for f in ["fastopic_original.csv", "fastopic_existing.csv", "fastopic_constructed.csv"]])
res = res[res.K == 50]

def mean_sd(v):
    v = v.dropna()
    return "" if v.empty else f"{v.mean():.3f}" if len(v) == 1 else f"{v.mean():.3f} ± {v.std():.3f}"

table = res.groupby("dataset")[M6].agg(mean_sd)
table.insert(0, "seeds", res.groupby("dataset").seed.nunique())
order = [(g, n) for g, names in GROUPS.items() for n in names]
table = table.loc[[n for _, n in order]]
table.index = pd.MultiIndex.from_tuples(order)
table

seeds             CV             TD  \
paper                     20NG              5  0.417 ± 0.007  0.993 ± 0.004   
                          NYT               5  0.427 ± 0.009  1.000 ± 0.001   
                          NeurIPS           5  0.429 ± 0.004  0.999 ± 0.001   
                          ACL               5  0.411 ± 0.010  0.998 ± 0.002   
                          Wikitext-103      5  0.430 ± 0.007  0.998 ± 0.002   
replaces WoS              DBpedia           5  0.465 ± 0.010  0.973 ± 0.025   
existing, new to FASTopic BBC               5  0.451 ± 0.007  1.000 ± 0.001   
                          OHSUMED           5  0.511 ± 0.013  1.000 ± 0.000   
                          Reuters           5  0.435 ± 0.007  0.950 ± 0.022   
                          TREC-coarse       5  0.308 ± 0.037  0.099 ± 0.038   
                          TREC-fine         5  0.308 ± 0.037  0.099 ± 0.038   
built by us               Guardian          5  0.431 ± 0.008  0.998 ± 0.001   
                          TMDB              5  0.445 ± 0.003  0.992 ± 0.008   

                                               Purity            NMI  \
paper                     20NG          0.587 ± 0.015  0.528 ± 0.004   
                          NYT           0.664 ± 0.006  0.371 ± 0.003   
                          NeurIPS                                      
                          ACL                                          
                          Wikitext-103                                 
replaces WoS              DBpedia       0.847 ± 0.009  0.690 ± 0.010   
existing, new to FASTopic BBC           0.909 ± 0.009  0.502 ± 0.008   
                          OHSUMED       0.456 ± 0.007  0.319 ± 0.006   
                          Reuters       0.766 ± 0.015  0.478 ± 0.013   
                          TREC-coarse   0.481 ± 0.023  0.208 ± 0.010   
                          TREC-fine     0.371 ± 0.008  0.307 ± 0.004   
built by us               Guardian      0.787 ± 0.005  0.428 ± 0.005   
                          TMDB          0.675 ± 0.011  0.322 ± 0.009   

                                                  Acc             F1  
paper                     20NG          0.694 ± 0.003  0.680 ± 0.002  
                          NYT           0.777 ± 0.006  0.590 ± 0.005  
                          NeurIPS                                     
                          ACL                                         
                          Wikitext-103                                
replaces WoS              DBpedia       0.946 ± 0.002  0.947 ± 0.002  
existing, new to FASTopic BBC           0.953 ± 0.003  0.952 ± 0.003  
                          OHSUMED       0.650 ± 0.003  0.520 ± 0.010  
                          Reuters       0.869 ± 0.002  0.300 ± 0.011  
                          TREC-coarse   0.455 ± 0.066  0.353 ± 0.044  
                          TREC-fine     0.316 ± 0.092  0.047 ± 0.016  
built by us               Guardian      0.858 ± 0.004  0.841 ± 0.004  
                          TMDB          0.748 ± 0.018  0.702 ± 0.019

**Notes:**
- Clearly separated categories work well. BBC and DBpedia have the highest Purity (0.85–0.91) and Acc (0.95); our Guardian dataset follows (Purity 0.79, Acc 0.86). Both datasets we built behave like the existing ones, with TD ≥ 0.99.
- Very short texts break FASTopic. TREC questions average 4 words after preprocessing: TD falls to 0.099 (most topics share the same words) and CV to 0.308. TREC-coarse and TREC-fine have identical CV and TD because they are the same texts and labels are not used in training.
- Many small classes lower macro F1, as with NYT in the replication. Reuters (58 labels) has Acc 0.869 but F1 0.300; TREC-fine (50 labels) has F1 0.047.
- Specialised vocabulary. OHSUMED has the highest CV (0.511) but low Purity (0.456): its 23 disease categories share much of their medical vocabulary.

## 5. Reruns
Seed 0 of each new dataset was run again (to save its topics for the LLM judge). Same check as in the replication notebook.

In [22]:
first = pd.concat([pd.read_csv(R / f) for f in ["fastopic_existing.csv", "fastopic_constructed.csv"]])
first = first.drop_duplicates(["dataset", "K", "seed"])
again = pd.read_csv(R / "rerun.csv").drop_duplicates(["dataset", "K", "seed"])
both = again.merge(first, on=["dataset", "K", "seed"], suffixes=(" rerun", " first"))
METRICS = ["TD", "Purity", "NMI", "Acc", "F1"]
both["max |diff|"] = pd.concat([(both[f"{m} rerun"] - both[f"{m} first"]).abs() for m in METRICS], axis=1).max(axis=1)
both.set_index(["dataset", "K", "seed"])[["max |diff|"]]

,,,max |diff|
dataset,K,seed,
Reuters,50,0,0.0
OHSUMED,50,0,0.0
TREC-coarse,50,0,0.0
TREC-fine,50,0,0.0
BBC,50,0,0.0
Guardian,50,0,0.0
TMDB,50,0,0.0
DBpedia,50,0,0.0


**Conclusion:** Every rerun gives the same numbers as the first run.